In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

def load_data(train_path, test_path):
    return pd.read_csv(train_path), pd.read_csv(test_path)

def preprocess_data(train_df, test_df):
    X_train, y_train = train_df.iloc[:, 1:-1].values, train_df.iloc[:, -1].values
    X_test = test_df.iloc[:, 1:].values[:, :X_train.shape[1]]

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Save label encoder and scaler
    joblib.dump(label_encoder, "label_encoder.pkl")
    joblib.dump(scaler, "scaler.pkl")

    return X_train_scaled[..., np.newaxis], y_train, X_test_scaled[..., np.newaxis], label_encoder

def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv1D(64, 5, padding='same', activation='relu', input_shape=input_shape),
        BatchNormalization(), MaxPooling1D(2),
        Conv1D(128, 3, padding='same', activation='relu'),
        BatchNormalization(), MaxPooling1D(2), Flatten(),
        Dense(128, activation='relu'), Dropout(0.4),
        Dense(64, activation='relu'), Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_model(model, X_train, y_train):
    return model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.2, verbose=1,
                     callbacks=[
                         ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
                         EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                     ])

def predict_labels(model, X_test, label_encoder, test_df):
    preds = label_encoder.inverse_transform(np.argmax(model.predict(X_test), axis=1))
    for filename, label in zip(test_df.iloc[:, 0], preds):
        print(f"📌 **{filename} → {label}**")

if __name__ == "__main__":
    train_df, test_df = load_data("/content/trainset_updated.csv", "/content/testset_updated.csv")
    X_train, y_train, X_test, label_encoder = preprocess_data(train_df, test_df)

    model = build_cnn((X_train.shape[1], 1), len(np.unique(y_train)))
    history = train_model(model, X_train, y_train)

    print(f"✅ Training: {history.history['accuracy'][-1]:.4f} | Validation: {history.history['val_accuracy'][-1]:.4f}")

    # Save model
    model.save("audio_cnn_model.h5")

    # Predict
    predict_labels(model, X_test, label_encoder, test_df)


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 178ms/step - accuracy: 0.3741 - loss: 1.6476 - val_accuracy: 0.5000 - val_loss: 1.0752 - learning_rate: 0.0010
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5651 - loss: 0.9171 - val_accuracy: 0.7000 - val_loss: 1.0410 - learning_rate: 0.0010
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8090 - loss: 0.3477 - val_accuracy: 1.0000 - val_loss: 1.0180 - learning_rate: 0.0010
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8993 - loss: 0.3070 - val_accuracy: 0.9000 - val_loss: 0.9973 - learning_rate: 0.0010
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8681 - loss: 0.3752 - val_accuracy: 0.9000 - val_loss: 0.9824 - learning_rate: 0.0010
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8602 - loss: 0.3468 - val_accuracy: 0.9000 - val_loss: 0.9742 - learning_rate: 0.0010
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8073 - loss: 0.4129 - val_accuracy: 0.9000 - val_loss

✅ Training: 0.9722 | Validation: 0.7000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
📌 **Audio_14.wav → Lion**
📌 **Audio_47.wav → Bear**
📌 **Audio_32.wav → Bear**
📌 **Audio_6.wav → Elephant**
📌 **Audio_28.wav → Bear**
📌 **Audio_22.wav → Elephant**
📌 **Audio_20.wav → Bear**
📌 **Audio_17.wav → Lion**
📌 **Audio_12.wav → Lion**


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

def load_data(train_path, test_path):
    return pd.read_csv(train_path), pd.read_csv(test_path)

def preprocess_data(train_df, test_df):
    X_train, y_train = train_df.iloc[:, 1:-1].values, train_df.iloc[:, -1].values
    X_test = test_df.iloc[:, 1:].values[:, :X_train.shape[1]]

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    joblib.dump(label_encoder, "label_encoder.pkl")
    joblib.dump(scaler, "scaler.pkl")

    return X_train_scaled[..., np.newaxis], y_train, X_test_scaled[..., np.newaxis], label_encoder

def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv1D(128, 7, padding='same', activation='relu', input_shape=input_shape),
        BatchNormalization(), MaxPooling1D(2),
        Conv1D(256, 5, padding='same', activation='relu'),
        BatchNormalization(), MaxPooling1D(2),
        Conv1D(128, 3, padding='same', activation='relu'),
        BatchNormalization(), MaxPooling1D(2),
        Flatten(),
        Dense(256, activation='relu'), Dropout(0.5),
        Dense(128, activation='relu'), Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(0.0005), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_model(model, X_train, y_train):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
    history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=50, batch_size=16, verbose=1,
                        callbacks=[
                            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
                            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                        ])
    return history

def predict_labels(model, X_test, label_encoder, test_df):
    preds = label_encoder.inverse_transform(np.argmax(model.predict(X_test), axis=1))
    for filename, label in zip(test_df.iloc[:, 0], preds):
        print(f"📌 {filename} → {label}")

if __name__ == "__main__":
    train_df, test_df = load_data("/content/train.1.csv", "/content/test.1.csv")
    X_train, y_train, X_test, label_encoder = preprocess_data(train_df, test_df)

    model = build_cnn((X_train.shape[1], 1), len(np.unique(y_train)))
    history = train_model(model, X_train, y_train)

    print(f"✅ Training Accuracy: {history.history['accuracy'][-1]:.4f} | Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

    model.save("audio_cnn_model.h5")

    predict_labels(model, X_test, label_encoder, test_df)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.0573 - loss: 8.1311 - val_accuracy: 0.3333 - val_loss: 1.7617 - learning_rate: 5.0000e-04
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.4062 - loss: 8.6698 - val_accuracy: 0.1667 - val_loss: 1.8499 - learning_rate: 5.0000e-04
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.6641 - loss: 5.7261 - val_accuracy: 0.0000e+00 - val_loss: 2.1644 - learning_rate: 5.0000e-04
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7083 - loss: 4.5194
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.7031 - loss: 4.6238 - val_accuracy: 0.0000e+00 - val_loss: 2.8318 - learning_rate: 5.0000e-04
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.6198 - loss: 4.7051 - val_accuracy: 0.0000e+00 - val_loss: 3.6589 - learning_rate: 2.5000e-04
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.7630 - loss: 3.5931 - va

✅ Training Accuracy: 0.7917 | Validation Accuracy: 0.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 529ms/step
📌 -172.18196 → Elephant
📌 -493.83466 → Lion
📌 -570.26654 → Elephant
📌 -381.99597 → Elephant
📌 -669.8628 → Elephant
📌 -764.58545 → Elephant
📌 -875.0794 → Elephant
📌 -421.13297 → Elephant
📌 -682.76086 → Elephant
📌 -377.45422 → Elephant
📌 -355.46814 → Elephant
📌 -529.45105 → Elephant
📌 -509.591 → Elephant
📌 -387.81372 → Elephant
📌 -740.6802 → Cheetah


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

def load_data(train_path, test_path):
    return pd.read_csv(train_path), pd.read_csv(test_path)

def preprocess_data(train_df, test_df):
    X_train = train_df.iloc[:, 1:-1].values
    y_train = train_df.iloc[:, -1].values
    X_test = test_df.iloc[:, 1:].values[:, :X_train.shape[1]]

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    joblib.dump(label_encoder, "label_encoder.pkl")
    joblib.dump(scaler, "scaler.pkl")

    return X_train_scaled[..., np.newaxis], y_train_encoded, X_test_scaled[..., np.newaxis], label_encoder

def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv1D(64, 5, padding='same', activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Conv1D(128, 3, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling1D(2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(0.0005), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_model(model, X_train, y_train):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
    )

    # Compute class weights
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    class_weights_dict = dict(enumerate(class_weights))

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        class_weight=class_weights_dict,
        callbacks=[
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        ],
        verbose=1
    )

    return history

def predict_labels(model, X_test, label_encoder, test_df):
    predictions = np.argmax(model.predict(X_test), axis=1)
    labels = label_encoder.inverse_transform(predictions)
    for fname, label in zip(test_df.iloc[:, 0], labels):
        print(f"📌 {fname} → {label}")

if __name__ == "__main__":
    train_df, test_df = load_data("/content/train.1.csv", "/content/test.1.csv")
    X_train, y_train, X_test, label_encoder = preprocess_data(train_df, test_df)

    model = build_cnn((X_train.shape[1], 1), len(np.unique(y_train)))
    history = train_model(model, X_train, y_train)

    print(f"✅ Training Accuracy: {history.history['accuracy'][-1]:.4f} | Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

    model.save("audio_cnn_model.h5")

    predict_labels(model, X_test, label_encoder, test_df)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.1667 - loss: 10.1410 - val_accuracy: 0.0833 - val_loss: 2.2928 - learning_rate: 5.0000e-04
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 668ms/step - accuracy: 0.2474 - loss: 23.5882 - val_accuracy: 0.1667 - val_loss: 2.3124 - learning_rate: 5.0000e-04
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 980ms/step - accuracy: 0.3385 - loss: 29.2893 - val_accuracy: 0.1667 - val_loss: 2.1933 - learning_rate: 5.0000e-04
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 784ms/step - accuracy: 0.2995 - loss: 17.1216 - val_accuracy: 0.4167 - val_loss: 1.8199 - learning_rate: 5.0000e-04
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 959ms/step - accuracy: 0.4635 - loss: 19.0098 - val_accuracy: 0.4167 - val_loss: 1.7386 - learning_rate: 5.0000e-04
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 629ms/step - accuracy: 0.4974 - loss: 23.5804 - val_accuracy: 0.3333 - val_loss: 1.6769 - learning_rate: 5.0000e-04
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 603ms/step - accuracy: 0.5573 - lo

✅ Training Accuracy: 0.5625 | Validation Accuracy: 0.3333
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
📌 -172.18196 → Elephant
📌 -493.83466 → Lion
📌 -570.26654 → Bear
📌 -381.99597 → Goat
📌 -669.8628 → Buffalo
📌 -764.58545 → Lion
📌 -875.0794 → Elephant
📌 -421.13297 → Elephant
📌 -682.76086 → Buffalo
📌 -377.45422 → Bear
📌 -355.46814 → Lion
📌 -529.45105 → Bear
📌 -509.591 → Bear
📌 -387.81372 → Lion
📌 -740.6802 → Buffalo


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

def load_data(train_path, test_path):
    return pd.read_csv(train_path), pd.read_csv(test_path)

def preprocess_data(train_df, test_df):
    X_train = train_df.iloc[:, 1:-1].values
    y_train = train_df.iloc[:, -1].values
    X_test = test_df.iloc[:, 1:].values[:, :X_train.shape[1]]

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)

    # One-hot encode the labels
    onehot_encoder = OneHotEncoder(sparse_output=False)
    y_train_oh = onehot_encoder.fit_transform(y_train_encoded.reshape(-1, 1))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Save encoders and scaler
    joblib.dump(label_encoder, "label_encoder.pkl")
    joblib.dump(scaler, "scaler.pkl")

    return X_train_scaled[..., np.newaxis], y_train_oh, X_test_scaled[..., np.newaxis], label_encoder

def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv1D(64, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.0003),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_model(model, X_train, y_train):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=np.argmax(y_train, axis=1), random_state=42
    )

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=[
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
            EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
        ],
        verbose=1
    )
    return history

def predict_labels(model, X_test, label_encoder, test_df):
    preds = np.argmax(model.predict(X_test), axis=1)
    labels = label_encoder.inverse_transform(preds)
    for filename, label in zip(test_df.iloc[:, 0], labels):
        print(f"📌 {filename} → {label}")

if __name__ == "__main__":
    # Update file paths if needed
    train_df, test_df = load_data("/content/train.1.csv", "/content/test.1.csv")
    X_train, y_train, X_test, label_encoder = preprocess_data(train_df, test_df)

    model = build_cnn((X_train.shape[1], 1), y_train.shape[1])
    history = train_model(model, X_train, y_train)

    print(f"✅ Training Accuracy: {history.history['accuracy'][-1]:.4f} | Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

    # Save model
    model.save("audio_cnn_model.keras")

    # Predict and display results
    predict_labels(model, X_test, label_encoder, test_df)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 835ms/step - accuracy: 0.2396 - loss: 5.6139 - val_accuracy: 0.3333 - val_loss: 1.4538 - learning_rate: 3.0000e-04
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 649ms/step - accuracy: 0.5938 - loss: 3.9909 - val_accuracy: 0.5833 - val_loss: 1.2214 - learning_rate: 3.0000e-04
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 634ms/step - accuracy: 0.5573 - loss: 1.5259 - val_accuracy: 0.5000 - val_loss: 1.4832 - learning_rate: 3.0000e-04
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 992ms/step - accuracy: 0.7969 - loss: 2.0569 - val_accuracy: 0.5833 - val_loss: 1.3188 - learning_rate: 3.0000e-04
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 724ms/step - accuracy: 0.9453 - loss: 0.1857 - val_accuracy: 0.6667 - val_loss: 1.0187 - learning_rate: 3.0000e-04
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 629ms/step - accuracy: 0.9271 - loss: 0.8621 - val_accuracy: 0.7500 - val_loss: 0.8950 - learning_rate: 3.0000e-04
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 628ms/step - accuracy: 1.0000 - loss:

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Load data
def load_data(train_path, test_path):
    return pd.read_csv(train_path), pd.read_csv(test_path)

# Preprocess data
def preprocess_data(train_df, test_df):
    X_train = train_df.iloc[:, 1:-1].values
    y_train = train_df.iloc[:, -1].values
    X_test = test_df.iloc[:, 1:].values[:, :X_train.shape[1]]

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)

    # One-hot encode the labels
    onehot_encoder = OneHotEncoder(sparse_output=False)
    y_train_oh = onehot_encoder.fit_transform(y_train_encoded.reshape(-1, 1))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Save encoders and scaler for future use
    joblib.dump(label_encoder, "label_encoder.pkl")
    joblib.dump(scaler, "scaler.pkl")

    return X_train_scaled[..., np.newaxis], y_train_oh, X_test_scaled[..., np.newaxis], label_encoder

# Build CNN Model
def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv1D(64, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.0003),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Train Model with EarlyStopping and ReduceLROnPlateau
def train_model(model, X_train, y_train):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=np.argmax(y_train, axis=1), random_state=42
    )

    # Define callbacks
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )
    return history

# Predict and print labels
def predict_labels(model, X_test, label_encoder, test_df):
    preds = np.argmax(model.predict(X_test), axis=1)
    labels = label_encoder.inverse_transform(preds)
    for filename, label in zip(test_df.iloc[:, 0], labels):
        print(f"📌 {filename} → {label}")

# Main function
if __name__ == "__main__":
    # Update file paths if needed
    train_df, test_df = load_data("/content/train.1.csv", "/content/test.1.csv")
    X_train, y_train, X_test, label_encoder = preprocess_data(train_df, test_df)

    # Build and train the CNN model
    model = build_cnn((X_train.shape[1], 1), y_train.shape[1])
    history = train_model(model, X_train, y_train)

    # Log final training and validation accuracy
    print(f"✅ Training Accuracy: {history.history['accuracy'][-1]:.4f} | Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

    # Save model and training history
    model.save("audio_cnn_model.keras")
    joblib.dump(history.history, "training_history.pkl")

    # Predict and display results
    predict_labels(model, X_test, label_encoder, test_df)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 918ms/step - accuracy: 0.3203 - loss: 7.3566 - val_accuracy: 0.4167 - val_loss: 1.4949 - learning_rate: 3.0000e-04
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.6172 - loss: 4.7108 - val_accuracy: 0.5833 - val_loss: 1.3527 - learning_rate: 3.0000e-04
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 668ms/step - accuracy: 0.6953 - loss: 2.9817 - val_accuracy: 0.7500 - val_loss: 1.1135 - learning_rate: 3.0000e-04
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 645ms/step - accuracy: 0.8385 - loss: 1.6361 - val_accuracy: 0.6667 - val_loss: 1.0951 - learning_rate: 3.0000e-04
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 656ms/step - accuracy: 0.8984 - loss: 0.2891 - val_accuracy: 0.5000 - val_loss: 1.0756 - learning_rate: 3.0000e-04
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 825ms/step - accuracy: 0.8151 - loss: 1.7968 - val_accuracy: 0.5833 - val_loss: 1.0280 - learning_rate: 3.0000e-04
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 891ms/step - accuracy: 0.9531 - loss: 0.

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step
📌 -172.18196 → Bear
📌 -493.83466 → Bear
📌 -570.26654 → Bear
📌 -381.99597 → Bear
📌 -669.8628 → Bear
📌 -764.58545 → Bear
📌 -875.0794 → Cheetah
📌 -421.13297 → Elephant
📌 -682.76086 → Bear
📌 -377.45422 → Bear
📌 -355.46814 → Lion
📌 -529.45105 → Bear
📌 -509.591 → Bear
📌 -387.81372 → Bear
📌 -740.6802 → Buffalo


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras import regularizers

def load_data(train_path, test_path):
    return pd.read_csv(train_path), pd.read_csv(test_path)

def preprocess_data(train_df, test_df):
    X_train = train_df.iloc[:, 1:-1].values
    y_train = train_df.iloc[:, -1].values
    X_test = test_df.iloc[:, 1:].values[:, :X_train.shape[1]]

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)

    # One-hot encode the labels
    onehot_encoder = OneHotEncoder(sparse_output=False)
    y_train_oh = onehot_encoder.fit_transform(y_train_encoded.reshape(-1, 1))

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Save encoders and scaler
    joblib.dump(label_encoder, "label_encoder.pkl")
    joblib.dump(scaler, "scaler.pkl")

    return X_train_scaled[..., np.newaxis], y_train_oh, X_test_scaled[..., np.newaxis], label_encoder

def build_cnn(input_shape, num_classes):
    model = Sequential([
        Conv1D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.4),  # Increased dropout to prevent overfitting

        Conv1D(64, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.4),  # Increased dropout to prevent overfitting

        Flatten(),
        Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)),  # L2 Regularization
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001),  # Lower learning rate
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_model(model, X_train, y_train):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=np.argmax(y_train, axis=1), random_state=42
    )

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=16,
        callbacks=[
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
            EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
        ],
        verbose=1
    )
    return history

def predict_labels(model, X_test, label_encoder, test_df):
    preds = np.argmax(model.predict(X_test), axis=1)
    labels = label_encoder.inverse_transform(preds)
    for filename, label in zip(test_df.iloc[:, 0], labels):
        print(f"📌 {filename} → {label}")

if __name__ == "__main__":
    # Update file paths if needed
    train_df, test_df = load_data("/content/train.1.csv", "/content/test.1.csv")
    X_train, y_train, X_test, label_encoder = preprocess_data(train_df, test_df)

    model = build_cnn((X_train.shape[1], 1), y_train.shape[1])
    history = train_model(model, X_train, y_train)

    print(f"✅ Training Accuracy: {history.history['accuracy'][-1]:.4f} | Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

    # Save model
    model.save("audio_cnn_model.keras")

    # Predict and display results
    predict_labels(model, X_test, label_encoder, test_df)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 445ms/step - accuracy: 0.0938 - loss: 5.8823 - val_accuracy: 0.3333 - val_loss: 3.1417 - learning_rate: 1.0000e-04
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 269ms/step - accuracy: 0.4141 - loss: 3.2521 - val_accuracy: 0.4167 - val_loss: 3.1503 - learning_rate: 1.0000e-04
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 276ms/step - accuracy: 0.3542 - loss: 3.7497 - val_accuracy: 0.4167 - val_loss: 3.0849 - learning_rate: 1.0000e-04
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 484ms/step - accuracy: 0.5495 - loss: 2.4700 - val_accuracy: 0.5833 - val_loss: 2.9880 - learning_rate: 1.0000e-04
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 362ms/step - accuracy: 0.6224 - loss: 2.6755 - val_accuracy: 0.6667 - val_loss: 2.9318 - learning_rate: 1.0000e-04
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 262ms/step - accuracy: 0.6484 - loss: 2.4682 - val_accuracy: 0.6667 - val_loss: 2.8921 - learning_rate: 1.0000e-04
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 268ms/step - accuracy: 0.8229 - loss:

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
📌 -172.18196 → Dog
📌 -493.83466 → Dog
📌 -570.26654 → Bear
📌 -381.99597 → Dog
📌 -669.8628 → Dog
📌 -764.58545 → Dog
📌 -875.0794 → Dog
📌 -421.13297 → Elephant
📌 -682.76086 → Buffalo
📌 -377.45422 → Cheetah
📌 -355.46814 → Lion
📌 -529.45105 → Bear
📌 -509.591 → Bear
📌 -387.81372 → Dog
📌 -740.6802 → Goat
